# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
# Import necessary libraries
import pandas as pd  # Data processing: https://pandas.pydata.org/docs/
#import matplotlib.pyplot as plt  # Visualization: https://matplotlib.org/stable/contents.html
#import seaborn as sns  # Also used for visualization: https://seaborn.pydata.org/
import os
import io
import requests

def load_or_download_csv(url, filename, save_dir="data", delimiter=';', encoding=None):
    """
    Checks if the CSV file exists locally and validates it against the server
    using HTTP conditional requests (If-None-Match / ETag).
    """
    os.makedirs(save_dir, exist_ok=True)
    file_path = os.path.join(save_dir, filename)
    etag_path = file_path + ".etag"
    download_url = url.rstrip('/') + '/' + filename

    # 1. Read stored ETag if it exists locally
    saved_etag = None
    if os.path.exists(etag_path) and os.path.exists(file_path):
        try:
            with open(etag_path, 'r', encoding='utf-8') as f:
                saved_etag = f.read().strip()
        except Exception:
            pass

    # 2. Prepare headers for conditional request
    headers = {}
    if saved_etag:
        headers["If-None-Match"] = saved_etag

    try:
        print(f"Checking server for updates: {download_url}")
        response = requests.get(download_url, headers=headers)

        # Case A: File hasn't changed on the server
        if response.status_code == 304:
            print(f"File is up to date (304 Not Modified). Loading local file from: {file_path}")
            return _read_csv_with_encoding(file_path, delimiter, encoding)

        # Case B: File is new or has been updated on the server
        elif response.status_code == 200:
            print(f"New or updated file found. Downloading...")

            # Use BytesIO to pass downloaded content into your encoding-aware reader
            content_stream = io.BytesIO(response.content)
            df = _read_csv_with_encoding(content_stream, delimiter, encoding)

            # Save the CSV file locally
            df.to_csv(file_path, sep=delimiter, index=False, encoding='utf-8')

            # Save the new ETag provided by the server
            new_etag = response.headers.get("ETag")
            if new_etag:
                with open(etag_path, 'w', encoding='utf-8') as f:
                    f.write(new_etag)

            print(f"File successfully updated and saved to: {file_path}")
            return df

        else:
            print(f"Unexpected HTTP status code ({response.status_code}). Falling back to local file if available.")

    except requests.exceptions.RequestException as e:
        print(f"Network error or connection failed ({e}). Falling back to local cache...")

    # 3. Fallback: If network request failed or returned unexpected status, load local copy if it exists
    if os.path.exists(file_path):
        try:
            print(f"Loading fallback local file from: {file_path}")
            return _read_csv_with_encoding(file_path, delimiter, encoding)
        except Exception as local_e:
            print(f"Error reading local file: {local_e}")
    else:
        print("Error: No local file available and network request failed.")

    return None

def _read_csv_with_encoding(path_or_buffer, delimiter, encoding):
    """Helper function to try multiple encodings if the default fails."""
    if encoding:
        return pd.read_csv(path_or_buffer, delimiter=delimiter, encoding=encoding)

    # Try common encodings for European datasets
    encodings_to_try = ['utf-8', 'utf-8-sig', 'latin-1', 'iso-8859-1', 'cp1252']
    for enc in encodings_to_try:
        try:
            # If path_or_buffer is a stream, ensure we reset its pointer if retrying
            if hasattr(path_or_buffer, 'seek'):
                path_or_buffer.seek(0)
            return pd.read_csv(path_or_buffer, delimiter=delimiter, encoding=enc)
        except UnicodeDecodeError:
            continue

    if hasattr(path_or_buffer, 'seek'):
        path_or_buffer.seek(0)
    return pd.read_csv(path_or_buffer, delimiter=delimiter, encoding='utf-8')

### Download station names

In [2]:
pollen_url = 'https://data.geo.admin.ch/ch.meteoschweiz.ogd-pollen/'
file = "ogd-pollen_meta_stations.csv"
df_pollen_stations = load_or_download_csv(pollen_url, file, save_dir="data/meta")
# if df_pollen_stations is not None:
#     print(df_pollen_stations.head())

meteorological_url = 'https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/'
file = 'ogd-smn_meta_stations.csv'
df_meteo_stations = load_or_download_csv(meteorological_url, file, save_dir="data/meta")
# if df_meteo_stations is not None:
#     print(df_meteo_stations.head())

phenology_url = 'https://data.geo.admin.ch/ch.meteoschweiz.ogd-phenology/'
file = 'ogd-phenology_meta_stations.csv'
df_phenology_stations = load_or_download_csv(phenology_url, file, save_dir="data/meta")
if df_phenology_stations is not None:
    print(df_phenology_stations.head())


Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-pollen/ogd-pollen_meta_stations.csv
File is up to date (304 Not Modified). Loading local file from: data/meta/ogd-pollen_meta_stations.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/ogd-smn_meta_stations.csv
File is up to date (304 Not Modified). Loading local file from: data/meta/ogd-smn_meta_stations.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-phenology/ogd-phenology_meta_stations.csv
File is up to date (304 Not Modified). Loading local file from: data/meta/ogd-phenology_meta_stations.csv
  station_abbr  station_name station_canton station_wigos_id  \
0          ABT  L'Abergement             VD              NaN   
1          ADB     Adelboden             BE              NaN   
2          ALC    Alchenflüh             BE      0-756-0-ALC   
3          ALD       Altdorf             UR              NaN   
4          ANC       Altanca          

### Calculate nearest stations

In [3]:
from scipy.spatial import cKDTree

# 1. Extract LV95 coordinates (East, North) in meters
pollen_coords = df_pollen_stations[
    ["station_coordinates_lv95_east", "station_coordinates_lv95_north"]
].values
meteo_coords = df_meteo_stations[
    ["station_coordinates_lv95_east", "station_coordinates_lv95_north"]
].values
phenology_coords = df_phenology_stations[
    ["station_coordinates_lv95_east", "station_coordinates_lv95_north"]
].values

k = 1

# --- FIND CLOSEST METEO STATION FOR EACH POLLEN STATION ---
# Build tree from meteo stations (the pool of candidates)
meteo_tree = cKDTree(meteo_coords)
meteo_distances, meteo_indices = meteo_tree.query(pollen_coords, k=k)

# Attach results to pollen dataframe
df_pollen_stations["closest_meteo_station"] = df_meteo_stations.iloc[meteo_indices][
    "station_abbr"
].values
df_pollen_stations["meteo_distance_km"] = meteo_distances / 1000.0  # Convert to kilometers


# --- FIND CLOSEST PHENOLOGY STATION FOR EACH POLLEN STATION ---
# Build tree from phenology stations (the pool of candidates)
phenology_tree = cKDTree(phenology_coords)
phenology_distances, phenology_indices = phenology_tree.query(pollen_coords, k=k)

# Attach results to pollen dataframe
df_pollen_stations["closest_phenology_station"] = df_phenology_stations.iloc[phenology_indices][
    "station_abbr"
].values
df_pollen_stations["phenology_distance_km"] = phenology_distances / 1000.0  # Convert to kilometers


# View the matching results
print(
    df_pollen_stations[
        ["station_name", "closest_meteo_station", "meteo_distance_km", "closest_phenology_station", "phenology_distance_km"]
    ].to_string()
)

         station_name closest_meteo_station  meteo_distance_km closest_phenology_station  phenology_distance_km
0                Bern                   BER           5.400562                       SEL              10.941134
1               Basel                   BAS           2.297147                       BAB               1.472498
2           Buchs, SG                   VAD           6.128038                       BUH               1.708508
3   La Chaux-de-Fonds                   CDF           4.541528                       LOL               9.373867
4    Davos / Wolfgang                   DAV           2.009922                       DAD               2.659777
5              Genève                   GVE           6.361877                       CAR              10.429774
6     Locarno / Monti                   OTL           0.033242                       LOM               0.002236
7            Lausanne                   PUY           2.181180                       ECD               7

### Download Pollen historical data

In [4]:
pollen_dfs = []
for station in df_pollen_stations.station_abbr:
    url = pollen_url + station.lower() + "/"
    file = f"ogd-pollen_{station.lower()}_d_historical.csv"
    df_station = load_or_download_csv(url, file, save_dir="data/pollen")
    if df_station is not None and not df_station.empty:
        pollen_dfs.append(df_station)

if pollen_dfs:
    df_pollen_data = pd.concat(pollen_dfs, ignore_index=True)
    print(f"Successfully combined data for {len(pollen_dfs)} stations. Total rows: {len(df_pollen_data)}")
else:
    df_pollen_data = pd.DataFrame()
    print("No data was loaded.")

print(df_pollen_data.head())

Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-pollen/pbe/ogd-pollen_pbe_d_historical.csv
File is up to date (304 Not Modified). Loading local file from: data/pollen/ogd-pollen_pbe_d_historical.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-pollen/pbs/ogd-pollen_pbs_d_historical.csv
File is up to date (304 Not Modified). Loading local file from: data/pollen/ogd-pollen_pbs_d_historical.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-pollen/pbu/ogd-pollen_pbu_d_historical.csv
File is up to date (304 Not Modified). Loading local file from: data/pollen/ogd-pollen_pbu_d_historical.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-pollen/pcf/ogd-pollen_pcf_d_historical.csv
File is up to date (304 Not Modified). Loading local file from: data/pollen/ogd-pollen_pcf_d_historical.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-pollen/pds/ogd-pollen

### Download Historical meteorological data

In [5]:
meteo_dfs = []
for station in df_pollen_stations.closest_meteo_station:
    url = meteorological_url + station.lower() + "/"
    file = f"ogd-smn_{station.lower()}_d_historical.csv"
    df_station = load_or_download_csv(url, file, save_dir="data/meteo")
    if df_station is not None and not df_station.empty:
        meteo_dfs.append(df_station)

if meteo_dfs:
    df_meteo_data = pd.concat(meteo_dfs, ignore_index=True)
    print(f"Successfully combined data for {len(meteo_dfs)} stations. Total rows: {len(df_meteo_data)}")
else:
    df_meteo_data = pd.DataFrame()
    print("No data was loaded.")

print(df_meteo_data.head())

Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/ber/ogd-smn_ber_d_historical.csv
File is up to date (304 Not Modified). Loading local file from: data/meteo/ogd-smn_ber_d_historical.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/bas/ogd-smn_bas_d_historical.csv
File is up to date (304 Not Modified). Loading local file from: data/meteo/ogd-smn_bas_d_historical.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/vad/ogd-smn_vad_d_historical.csv
File is up to date (304 Not Modified). Loading local file from: data/meteo/ogd-smn_vad_d_historical.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/cdf/ogd-smn_cdf_d_historical.csv
File is up to date (304 Not Modified). Loading local file from: data/meteo/ogd-smn_cdf_d_historical.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/dav/ogd-smn_dav_d_historical.csv
File is up to date (304 

### Download phenological data

In [16]:
phenology_dfs = []
for station in df_pollen_stations.closest_phenology_station:
    url = phenology_url + station.lower() + "/"
    file = f"ogd-phenology_{station.lower()}_y.csv"
    df_station = load_or_download_csv(url, file, save_dir="data/phenology")
    if df_station is not None and not df_station.empty:
        phenology_dfs.append(df_station)

if phenology_dfs:
    df_phenology_data = pd.concat(phenology_dfs, ignore_index=True)
    print(f"Successfully combined data for {len(phenology_dfs)} stations. Total rows: {len(df_phenology_data)}")
else:
    df_phenology_data = pd.DataFrame()
    print("No data was loaded.")

print(df_phenology_data.head())

Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-phenology/sel/ogd-phenology_sel_y.csv
New or updated file found. Downloading...
File successfully updated and saved to: data/phenology/ogd-phenology_sel_y.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-phenology/bab/ogd-phenology_bab_y.csv
New or updated file found. Downloading...
File successfully updated and saved to: data/phenology/ogd-phenology_bab_y.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-phenology/buh/ogd-phenology_buh_y.csv
New or updated file found. Downloading...
File successfully updated and saved to: data/phenology/ogd-phenology_buh_y.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteoschweiz.ogd-phenology/lol/ogd-phenology_lol_y.csv
New or updated file found. Downloading...
File successfully updated and saved to: data/phenology/ogd-phenology_lol_y.csv
Checking server for updates: https://data.geo.admin.ch/ch.meteos

# Starting Spark

In [3]:
# Konfiguracja projektu z importowaniem wymaganych bibliotek.
import shutil
from pyspark.sql import *
from pyspark.sql.types import *
from delta import *
from pathlib import Path

# JAVA_HOME jest wymagane dla prawidłowego importu biblioteki pyspark.
#if "JAVA_HOME" not in os.environ:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-25-openjdk"

builder = (
    SparkSession
        .builder
        .appName("Pollen")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

# Utworzenie sesji.
session = configure_spark_with_delta_pip(builder).getOrCreate()

# Konfiguracja dodatkowych parametrów sesji po jej utworzeniu.
session.conf.set("spark.sql.shuffle.partitions", "20")

:: loading settings :: url = jar:file:/opt/spark-4.2.0-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/Binny/.ivy2.5.2/cache
The jars for the packages stored in: /home/Binny/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b2d2051a-2cd3-4e91-aadc-daba189c4023;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found org.r

### Saving data

In [18]:
spark_master_df = (
    session.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .csv("data/pollen/*.csv")
)

print(f"Total partitions: {spark_master_df.rdd.getNumPartitions()}")
spark_master_df.printSchema()

(
    spark_master_df.write
    .format("delta")
    .mode("overwrite")
    .save("data/delta/pollen_historical")
)

/home/Binny/Projects/Pollen/.venv/lib64/python3.12/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/Binny/Projects/Pollen/.venv/lib64/python3.12/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/Binny/Projects/Pollen/.venv/lib64/python3.12/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Cannot convert pyarrow.lib.ChunkedArray to pyarrow.lib.Array
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is 

Successfully saved all pollen data to Delta table at: data/delta/pollen_historical


In [7]:
spark_master_df = (
    session.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .csv("data/meteo/*.csv")
)

print(f"Total partitions: {spark_master_df.rdd.getNumPartitions()}")
spark_master_df.printSchema()

(
    spark_master_df.write
    .format("delta")
    .mode("overwrite")
    .save("data/delta/meteo_historical")
)


26/10/25 23:16:26 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/meteo/*.csv.
java.io.FileNotFoundException: File data/meteo/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSo

Total partitions: 12
root
 |-- station_abbr: string (nullable = true)
 |-- reference_timestamp: string (nullable = true)
 |-- tre200d0: double (nullable = true)
 |-- tre200dx: double (nullable = true)
 |-- tre200dn: double (nullable = true)
 |-- tre005d0: double (nullable = true)
 |-- tre005dx: double (nullable = true)
 |-- tre005dn: double (nullable = true)
 |-- ure200d0: double (nullable = true)
 |-- pva200d0: double (nullable = true)
 |-- prestad0: double (nullable = true)
 |-- pp0qffd0: double (nullable = true)
 |-- ppz850d0: double (nullable = true)
 |-- ppz700d0: string (nullable = true)
 |-- pp0qnhd0: double (nullable = true)
 |-- fkl010d0: double (nullable = true)
 |-- fkl010d1: double (nullable = true)
 |-- fu3010d0: double (nullable = true)
 |-- fu3010d1: double (nullable = true)
 |-- fkl010d3: double (nullable = true)
 |-- fu3010d3: double (nullable = true)
 |-- wcc006d0: double (nullable = true)
 |-- rre150d0: double (nullable = true)
 |-- rka150d0: double (nullable = true)

26/10/25 23:16:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/25 23:16:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/25 23:16:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/10/25 23:16:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/10/25 23:16:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/10/25 23:16:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/10/25 23:16:31 WARN MemoryManager: Total allocation exceeds 95.

In [8]:
spark_master_df = (
    session.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .csv("data/phenology/*.csv")
)

print(f"Total partitions: {spark_master_df.rdd.getNumPartitions()}")
spark_master_df.printSchema()

(
    spark_master_df.write
    .format("delta")
    .mode("overwrite")
    .save("data/delta/phenology")
)

26/10/25 23:16:46 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/phenology/*.csv.
java.io.FileNotFoundException: File data/phenology/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis.Resol

Total partitions: 8
root
 |-- station_abbr: string (nullable = true)
 |-- reference_timestamp: string (nullable = true)
 |-- maesh13d: double (nullable = true)
 |-- maesh60d: double (nullable = true)
 |-- maesh65d: double (nullable = true)
 |-- maesh94d: double (nullable = true)
 |-- maesh95d: double (nullable = true)
 |-- mfags13d: double (nullable = true)
 |-- mfags94d: double (nullable = true)
 |-- mfags95d: double (nullable = true)
 |-- macep13d: double (nullable = true)
 |-- macep94d: double (nullable = true)
 |-- msora13d: double (nullable = true)
 |-- msora60d: double (nullable = true)
 |-- msora65d: double (nullable = true)
 |-- msora87d: double (nullable = true)
 |-- msora94d: double (nullable = true)
 |-- msora95d: double (nullable = true)
 |-- mcora13d: double (nullable = true)
 |-- mcora60d: double (nullable = true)
 |-- mcora65d: double (nullable = true)
 |-- mtilp13d: double (nullable = true)
 |-- mtilp60d: double (nullable = true)
 |-- mtilp65d: double (nullable = true)
